In [ ]:
# Cell 1

!pip -q install jax jaxlib scipy mpmath pillow numpy

In [ ]:
#Cell 2

import json
import sys, json, time
import numpy as np
import jax, jax.numpy as jnp
from scipy.optimize import minimize

jax.config.update("jax_enable_x64", True)

R = 1.0 / (2 * np.sin(np.pi / 8))      # circumradius of unit-side octagon
RIN = R * np.cos(np.pi / 8)             # inradius
S3 = np.sqrt(3.0)
NORM = jnp.array([[0.0, -1.0], [S3 / 2, 0.5], [-S3 / 2, 0.5]])
NPHI = jnp.array([-np.pi / 2, np.pi / 6, 5 * np.pi / 6])
K4 = jnp.arange(4) * (np.pi / 4)


def hsup(phi, th):
    d = jnp.mod(phi - th + np.pi / 8, np.pi / 4) - np.pi / 8
    return R * jnp.cos(d)


def unpack(z, n):
    return z[:n], z[n:2 * n], z[2 * n:3 * n], z[3 * n]


def cont_viol(z, n):
    x, y, th, s = unpack(z, n)
    H = jnp.array([0.0, s * S3 / 2, 0.0])
    c = jnp.stack([x, y], axis=1)                       # n,2
    proj = c @ NORM.T                                   # n,3
    h = hsup(NPHI[None, :], th[:, None])                # n,3
    return proj + h - H[None, :]                        # <=0 feasible


def pair_sep(z, n):
    x, y, th, s = unpack(z, n)
    dx = x[:, None] - x[None, :]
    dy = y[:, None] - y[None, :]
    # axes from octagon i (edge normals) : n,4
    phi_i = th[:, None] + np.pi / 8 + K4[None, :] # n,4
    phi_j = phi_i # same set, indexed by j
    def sep_from(phi_own_idx):
        # axes belonging to octagon 'a' (index along axis0), evaluated on pair matrix
        pass
    # axes of i: shape n,n,4
    A1 = phi_i[:, None, :] * jnp.ones((1, n, 1))
    A2 = phi_j[None, :, :] * jnp.ones((n, 1, 1))
    A = jnp.concatenate([A1, A2], axis=2)                # n,n,8
    ca, sa = jnp.cos(A), jnp.sin(A)
    pr = jnp.abs(ca * dx[:, :, None] + sa * dy[:, :, None])
    hi = hsup(A, th[:, None, None])
    hj = hsup(A, th[None, :, None])
    sep = pr - hi - hj
    return jnp.max(sep, axis=2)


def energy(z, n):
    cv = jnp.maximum(cont_viol(z, n), 0.0)
    ms = pair_sep(z, n)
    iu = jnp.triu(jnp.ones((n, n)), k=1)
    pen = jnp.maximum(-ms, 0.0) * iu
    return jnp.sum(cv ** 2) + jnp.sum(pen ** 2)


def make_fns(n):
    def obj(z, mu):
        return z[3 * n] + mu * energy(z, n)
    vg = jax.jit(jax.value_and_grad(obj))
    E = jax.jit(lambda z: energy(z, n))
    CV = jax.jit(lambda z: cont_viol(z, n))
    PS = jax.jit(lambda z: pair_sep(z, n))
    return vg, E, CV, PS


def max_violation(z, n, CV, PS):
    cv = float(jnp.max(CV(z)))
    ms = np.array(PS(z))
    iu = np.triu_indices(n, 1)
    return max(cv, float(-ms[iu].min())) if n > 1 else cv


def penalty_solve(z0, n, vg, mus=(1e1, 1e2, 1e3, 1e4, 1e5, 1e6), maxiter=400):
    z = z0.copy()
    for mu in mus:
        f = lambda zz: tuple(np.asarray(a, dtype=np.float64) for a in vg(zz, mu))
        r = minimize(f, z, jac=True, method="L-BFGS-B", options={"maxiter": maxiter, "maxcor": 30})
        z = r.x
    return z


def feasible_scale(z, n, CV, PS):
    x, y, th, s = unpack(jnp.asarray(z), n)
    lo, hi = 1.0, 1.0
    def viol(k):
        zz = jnp.concatenate([x * k, y * k, th, jnp.array([s * k])])
        return max_violation(zz, n, CV, PS)
    if viol(1.0) <= 0:
        return np.array(z)
    hi = 1.0 + 1e-6
    while viol(hi) > 0:
        hi = 1.0 + (hi - 1.0) * 2
        if hi > 2: return None
    lo = 1.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if viol(mid) > 0: lo = mid
        else: hi = mid
    k = hi
    return np.array(jnp.concatenate([x * k, y * k, th, jnp.array([s * k])]))


def slsqp_polish(z, n, CV, PS, iters=200):
    z = np.array(z)
    ms = np.array(PS(jnp.asarray(z)))
    x, y = z[:n], z[n:2 * n]
    D = np.hypot(x[:, None] - x[None, :], y[:, None] - y[None, :])
    I, J = np.where(np.triu(D < 2 * R + 0.8, 1))
    I = jnp.array(I); J = jnp.array(J)

    def cons(zz):
        zz = jnp.asarray(zz)
        cv = -cont_viol(zz, n).ravel()
        p = pair_sep(zz, n)[I, J]
        return jnp.concatenate([cv, p])
    cj = jax.jit(cons)
    cjac = jax.jit(jax.jacfwd(cons))
    res = minimize(lambda zz: zz[3 * n], z, jac=lambda zz: np.eye(1, 3 * n + 1, 3 * n)[0],
                   constraints=[{"type": "ineq", "fun": lambda zz: np.asarray(cj(zz)),
                                 "jac": lambda zz: np.asarray(cjac(zz))}],
                   method="SLSQP", options={"maxiter": iters, "ftol": 1e-13})
    return res.x


def random_init(n, s0, rng):
    pts = []
    while len(pts) < n:
        u, v = rng.random(2)
        if u + v > 1: u, v = 1 - u, 1 - v
        pts.append((s0 * (u + v * 0.5), s0 * v * S3 / 2))
    pts = np.array(pts)
    th = rng.random(n) * (np.pi / 4)
    return np.concatenate([pts[:, 0], pts[:, 1], th, [s0]])


def lattice_init(n, rng, jitter=0.15, aligned=True):
    w = 1 + np.sqrt(2)
    m = 1
    while m * (m + 1) // 2 < n: m += 1
    s0 = (m - 1) * w * 1.02 + 2 * RIN + 1.1 * w * 0.0 + 2.2
    pts = []
    rowh = w * np.sqrt(3) / 2 * 1.02
    for r in range(m):
        cnt = m - r
        y = RIN + r * rowh
        x0 = RIN * 1.0 + 0.5 * r * w * 1.02 + 0.6 * r * 0

        for k in range(cnt):
            pts.append((x0 + k * w * 1.02 + r * 0.6, y))
    pts = np.array(pts)

    if len(pts) > n:
        drop = rng.choice(len(pts), len(pts) - n, replace=False)
        keep = np.setdiff1d(np.arange(len(pts)), drop)
        pts = pts[keep]
    pts = pts + rng.normal(0, jitter, pts.shape)
    th = np.zeros(n) if aligned else rng.random(n) * np.pi / 4
    th = th + rng.normal(0, 0.05, n)
    s0 = pts[:, 0].max() * 1.6 + 6
    s0 = max(s0, 1.15 * (m - 1) * w + 8)
    return np.concatenate([pts[:, 0], pts[:, 1], th, [s0]])


def run(n, seconds, seed=0, verbose=True, out=None):
    rng = np.random.default_rng(seed)
    vg, E, CV, PS = make_fns(n)
    best = None
    t0 = time.time()
    trial = 0
    while time.time() - t0 < seconds:
        trial += 1
        if trial % 2 == 0:
            s_est = np.sqrt(n * 4.83 / (np.sqrt(3) / 4) / 0.85)
            z0 = random_init(n, s_est * 1.25, rng)
        else:
            z0 = lattice_init(n, rng)
        z = penalty_solve(z0, n, vg)
        zf = feasible_scale(z, n, CV, PS)
        if zf is None: continue
        if best is not None and zf[3 * n] > best[3 * n] + 0.15:
            continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            zp2 = feasible_scale(zp, n, CV, PS)
            if zp2 is not None and zp2[3 * n] < zf[3 * n]:
                zf = zp2
        except Exception as e:
            pass
        s = zf[3 * n]
        if best is None or s < best[3 * n]:
            best = zf
            if verbose:
                print(f"n={n} trial={trial} t={time.time()-t0:.0f}s s={s:.6f}", flush=True)
            if out:
                json.dump({"n": n, "s": float(s), "z": best.tolist()}, open(out, "w"))
    return best




In [ ]:
# Cell 3

import sys, json, time
import numpy as np, jax.numpy as jnp

def hop(n, seconds, seed, infile, out):
    rng = np.random.default_rng(seed)
    vg,E,CV,PS = make_fns(n)
    z = np.array(json.load(open(infile))["z"]); best = z.copy(); cur = z.copy()
    t0=time.time(); it=0
    while time.time()-t0 < seconds:
        it+=1
        zz = cur.copy()
        mode = rng.integers(3)
        k = rng.integers(1, max(2, n//4))
        idx = rng.choice(n, k, replace=False)
        sig = rng.choice([0.3, 0.7, 1.2])
        zz[idx] += rng.normal(0, sig, k)
        zz[n+idx] += rng.normal(0, sig, k)
        zz[2*n+idx] += rng.normal(0, 0.4, k)
        if mode == 0:
            j = rng.integers(n)
            s = zz[3*n]
            u,v = rng.random(2)
            if u+v>1: u,v=1-u,1-v
            zz[j] = s*(u+v*0.5); zz[n+j] = s*v*S3/2; zz[2*n+j]=rng.random()*np.pi/4
        zz[3*n] = cur[3*n]*1.04
        z1 = penalty_solve(zz, n, vg, mus=(1e2,1e3,1e4,1e5,1e6), maxiter=300)
        zf = feasible_scale(z1, n, CV, PS)
        if zf is None or zf[3*n] > best[3*n] + 0.08: continue
        try:
            zp = slsqp_polish(zf, n, CV, PS)
            z2 = feasible_scale(zp, n, CV, PS)
            if z2 is not None and z2[3*n] < zf[3*n]: zf = z2
        except Exception: pass
        s = zf[3*n]
        if s < cur[3*n] + 0.01: cur = zf
        if s < best[3*n] - 1e-9:
            best = zf; print(f"n={n} it={it} t={time.time()-t0:.0f}s s={s:.6f}", flush=True)
            json.dump({"n":n,"s":float(s),"z":best.tolist()}, open(out,"w"))
    return best



In [ ]:
# Cell 4

import mpmath as mp
mp.mp.dps = 60

def certify(z, n, eps=1e-9):
    z = [mp.mpf(float(v)) for v in z]; k = 1 + mp.mpf(eps)
    x = [v * k for v in z[:n]]; y = [v * k for v in z[n:2*n]]; th = z[2*n:3*n]; s = z[3*n] * k
    Rm = 1 / (2 * mp.sin(mp.pi / 8)); r3 = mp.sqrt(3)
    V = [[(x[i] + Rm*mp.cos(th[i] + j*mp.pi/4), y[i] + Rm*mp.sin(th[i] + j*mp.pi/4)) for j in range(8)] for i in range(n)]
    cont = min(min(vy, r3*vx - vy, r3*(s - vx) - vy) for P in V for vx, vy in P)
    def gap(A, B):
        best = -mp.inf
        for P in (A, B):
            for j in range(8):
                ex, ey = P[(j+1) % 8][0] - P[j][0], P[(j+1) % 8][1] - P[j][1]
                nx, ny = ey, -ex; nn = mp.sqrt(nx*nx + ny*ny)
                pa = [(px*nx + py*ny)/nn for px, py in A]; pb = [(px*nx + py*ny)/nn for px, py in B]
                best = max(best, min(pb) - max(pa), min(pa) - max(pb))
        return best
    pair = mp.inf
    for i in range(n):
        for j in range(i+1, n):
            if mp.hypot(x[i]-x[j], y[i]-y[j]) > 2*Rm + mp.mpf('1e-6'):   # circumcircles disjoint
                continue
            pair = min(pair, gap(V[i], V[j]))
    assert cont > 0 and pair > 0, (cont, pair)
    return s, cont, pair

def page_value(s):
    return f"{mp.floor(s * 10**5) / 10**5}+"


In [ ]:
# Cell 5

import os, shutil, csv, time
import numpy as np

KNOWN = {16: 15.66136, 17: 16.06191, 18: 16.40911, 19: 16.54447, 20: 16.74196,
         21: 16.78582, 22: 17.96238, 23: 18.39413, 24: 18.55965, 25: 18.93175}

def grow(zp, n, rng, tries=6):
    m = n - 1; vg, E, CV, PS = make_fns(n); best = None
    for _ in range(tries):
        x, y, th, s = zp[:m], zp[m:2*m], zp[2*m:3*m], zp[3*m]
        u, v = rng.random(2)
        if u + v > 1: u, v = 1 - u, 1 - v
        z0 = np.concatenate([x, [s*(u + v/2)], y, [s*v*S3/2], th, [rng.random()*np.pi/4], [s*1.05]])
        zf = feasible_scale(penalty_solve(z0, n, vg, mus=(1e2, 1e3, 1e4, 1e5, 1e6), maxiter=300), n, CV, PS)
        if zf is not None and (best is None or zf[3*n] < best[3*n]): best = zf
    return best

def geometry(z, n, W=250):
    s = z[3*n]; x, y, th = z[:n], z[n:2*n], z[2*n:3*n]
    pad = 1; sc = (W - 2*pad) / s; H = s*np.sqrt(3)/2*sc + 2*pad
    P = lambda px, py: (pad + px*sc, H - pad - py*sc)
    octs = [[P(x[i] + R*np.cos(th[i] + k*np.pi/4), y[i] + R*np.sin(th[i] + k*np.pi/4)) for k in range(8)] for i in range(n)]
    tri = [P(0, 0), P(s, 0), P(s/2, s*np.sqrt(3)/2)]
    return octs, tri, W, H

def write_svg(z, n, path):
    octs, tri, W, H = geometry(z, n)
    pts = lambda v: " ".join(f"{a:.3f},{b:.3f}" for a, b in v)
    body = "".join(f'<polygon points="{pts(o)}" fill="#ffffe7" stroke="#000" stroke-width="0.75"/>' for o in octs)
    open(path, "w").write(f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{int(H)}" viewBox="0 0 {W} {H:.3f}">'
                          f'{body}<polygon points="{pts(tri)}" fill="none" stroke="#000" stroke-width="1.5"/></svg>')


In [ ]:
# Cell 6

NS = range(25, 51)
SEARCH_SEC = 240       # random-restart time per n
HOP_SEC = 120          # basin hopping per n
OUT = "/content/octagons"; os.makedirs(OUT, exist_ok=True)

prev, rows = None, []
for n in NS:
    jf = f"{OUT}/{n}.json"
    if os.path.exists(jf):
        d = json.load(open(jf)); prev = np.array(d["z"]); rows.append((n, d["s_cert"])); continue
    rng = np.random.default_rng(n)
    cands = [c for c in (run(n, SEARCH_SEC, n, verbose=False), grow(prev, n, rng) if prev is not None else None) if c is not None]
    start = min(cands, key=lambda c: c[3*n])
    json.dump({"n": n, "s": float(start[3*n]), "z": start.tolist()}, open("/tmp/start.json", "w"))
    best = hop(n, HOP_SEC, n + 1000, "/tmp/start.json", "/tmp/hop.json")
    s_cert, cs, pg = certify(best, n)
    write_svg(best, n, f"{OUT}/{n}.svg");
    json.dump({"n": n, "s_cert": float(s_cert), "z": [float(v) for v in best]}, open(jf, "w"))
    rows.append((n, float(s_cert))); prev = best
    note = f"  page {KNOWN[n]}+ -> {'BEATS' if s_cert < KNOWN[n] else 'worse'}" if n in KNOWN else ""
    print(f"n={n}  certified s={mp.nstr(s_cert, 10)}  submit {page_value(s_cert)}{note}", flush=True)

with open(f"{OUT}/summary.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["n", "s_certified", "s_page_format"])
    for n, s in rows: w.writerow([n, s, page_value(mp.mpf(s))])
for (a, sa), (b, sb) in zip(rows, rows[1:]):
    if sb < sa: print(f"warning: s({b}) < s({a}); n={a} search is weak, rerun it longer")
shutil.make_archive("/content/octagons", "zip", OUT)
try:
    from google.colab import files; files.download("/content/octagons.zip")
except ImportError: pass